# L1b: Toolchain and Notebook Smoke Test

This lab verifies the complete course workflow end to end, then puts it to work: activate the shared environment, load local course code, visualize generated data, and write your first tested engineering calculation.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Run a notebook end to end:__ Execute a Julia notebook from top to bottom in the shared course environment. A clean run confirms that Julia, the notebook front end, and the pinned package set are all installed correctly on your machine.
> * __Confirm that packages and local code load:__ Verify that both external packages and the source files in `src/` are visible after the meeting-local `Include.jl` runs. This is what connects a notebook to the rest of the course.
> * __Translate a contract into a tested function:__ State the units and admissible inputs of an engineering calculation before writing it, implement it in a source file, and read the accompanying `Test` set as an executable statement of what it must do.

Let's get started!
___

## Setup, Data, and Prerequisites

In our case, our `Include.jl` file will set paths so our notebook knows where to find things, and then will load external packages into the global scope with [the `using` command](https://docs.julialang.org/en/v1/base/base/#using). This makes the content of the package visible to us. 

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Now that we have our environment setup, we can do some stuff.

___

## Task 1: Let's build and visualize a Normal Distribution
In this task, let's test our installation by sampling a model of [a Normal probability distribution](https://en.wikipedia.org/wiki/Normal_distribution), and then visualizing the samples. First, let's draw samples from the distribution and save them in the `samples::Array{Float64,1}` array. 

> __What is the `let` block?__ The [`let` block](https://docs.julialang.org/en/v1/base/base/#let) creates a new hard scope and optionally introduces new local bindings. Variables introduced inside a `let` block are local to that block and don't affect variables of the same name in the outer scope. In our case, the `let` block allows us to create local variables (`number_of_samples` and the local `samples`) that are only visible within the block, while the final value of `samples` is returned and assigned to the global variable `samples`. This is a common Julia pattern for organizing code and avoiding namespace pollution.

We use [the built-in `randn(...)` function](https://docs.julialang.org/en/v1.11/stdlib/Random/#Base.randn) to generate samples from a standard normal distribution (mean 0, variance 1).

In [ ]:
samples = let

    # initialize -
    number_of_samples = 10000; # set the number of samples we want to generate
    samples = randn(number_of_samples);

    samples; # return
end;

Now, let's plot the `sample::Array{Float64,1}` array using [the `histogram(...)` function exported by the `UnicodePlots.jl` package](https://juliaplots.org/UnicodePlots.jl/dev/api/#UnicodePlots.histogram-Tuple%7BAbstractArray%7D). 

In [ ]:
let

    # initialize -
    data = samples; # random array
    number_of_bins = 20; # how many bins?
    vertical = false; # vertical or horizontal?
    closed = :left; # which side of the interval is closed?

    # make the histogram -
    histogram(data, nbins=number_of_bins, vertical = vertical, closed = closed); 
end

In [ ]:
do_you_see_the_histogram = false; # TODO: set this to true once you actually see the histogram above

___

## Task 2: Can we see code that we wrote?
In this task, we check that methods that we wrote (that were included when we called the `Include.jl` file) are now visible. In [the `HelloWorld.jl` file](src/HelloWorld.jl), we defined [the `printgreeting()` function](src/HelloWorld.jl), which __returns__ the string `"Hello World!"`.

> __Return, not print:__ despite the name, `printgreeting()` has no side effect — it prints nothing. The notebook shows the string because the cell _displays the value of its last expression_. That distinction matters as soon as you start composing functions: a printed value is gone, a returned value can be used.

We'll save the returned value in the `message_that_we_get::String` variable:

In [ ]:
message_that_we_get = printgreeting() # returns "Hello World!"; the notebook displays it

___

## A first tested calculation

The toolchain works. Now use it for something.

An engineering calculation is a formula plus a statement of what its symbols mean and which values are admissible. Writing that contract down first is what turns a formula into something a test can check.

The [ideal gas law](https://en.wikipedia.org/wiki/Ideal_gas_law) relates the state variables of a gas:

$$
PV = nRT
$$

We want pressure, so rearrange for $P$:

$$
P = \frac{nRT}{V}
$$

> __The contract:__
>
> * $n$ — amount of substance, in mol. Finite and strictly positive.
> * $T$ — absolute temperature, in K. Finite and strictly positive; at or below absolute zero the model does not apply.
> * $V$ — volume, in $\mathrm{m^{3}}$. Finite and strictly positive.
> * $R$ — the [universal gas constant](https://en.wikipedia.org/wiki/Gas_constant), $8.31446261815324\;\mathrm{Pa\,m^{3}\,mol^{-1}\,K^{-1}}$. Exact by definition since the 2019 SI redefinition, so it is a default rather than an argument you supply.
> * Returns $P$ — pressure, in Pa.

[The `ideal_gas_pressure(...)` function](src/Compute.jl) lives in [`src/Compute.jl`](src/Compute.jl), which currently holds a signature, a docstring, and two `TODO` comments.

> __What to write:__
>
> * __Validate the inputs.__ Throw an [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError) naming the argument when any of them is not finite, or not strictly positive. [The `isfinite(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.isfinite) does the first check.
> * __Return the pressure.__ Compute $P = nRT/V$ and return it as a `Float64`.

Open the file, complete both `TODO`s, then restart the kernel and run this notebook from the top.

A new function deserves a case whose answer you already know. One mole at $273.15\;\mathrm{K}$ in about $22.414\;\mathrm{L}$ should come back near one atmosphere.

> __Where does $22.414\;\mathrm{L}$ come from?__
>
> It is not a measurement, it is a rearrangement: $V = nRT/P$ with $n = 1$, $T = 273.15\;\mathrm{K}$, and $P = 101325\;\mathrm{Pa}$. Feeding that volume back in should therefore return one atmosphere. This confirms the arithmetic inverts correctly; it is not independent evidence about any real gas.

We store the result in `pressure_Pa::Float64` and its kilopascal equivalent in `pressure_kPa::Float64`:

In [ ]:
amount_mol = 1.0
temperature_K = 273.15
volume_m3 = 0.02241396954
pressure_Pa = ideal_gas_pressure(amount_mol, temperature_K, volume_m3)
pressure_kPa = pressure_Pa / 1000
(pressure_Pa = pressure_Pa, pressure_kPa = pressure_kPa)

About $101$ kPa — one atmosphere, which is the number worth carrying in your head as a sanity check for gas-phase work.

___

___

## Tests
In the code block below, we check some values in your notebook and give you feedback on which items are correct or different. `Unhide` the code block below (if you are curious) about how we implemented the tests and what we are testing.

In [ ]:
let

    @testset verbose = true "CHEME 4/5800 L1b Test Suite" begin
        
        # Test 1: Environment Setup
        @testset "Environment Setup Tests" begin
            @test isdefined(Main, :CHEME5800_L1B_ROOT)
            @test CHEME5800_L1B_ROOT == @__DIR__
            @test @isdefined printgreeting
        end
        
        # Test 2: Sample Generation Tests
        @testset "Sample Generation Tests" begin
            @test @isdefined samples
            @test isa(samples, Array{Float64,1})
            @test length(samples) == 10000
            @test !isempty(samples)
            
            # Statistical properties of normal distribution
            sample_mean = Statistics.mean(samples)
            sample_std = Statistics.std(samples)
            @test abs(sample_mean) < 0.1
            @test abs(sample_std - 1.0) < 0.1
        end
        
        # Test 3: Histogram Flag Tests
        # This one fails until you look at the plot and flip the flag yourself.
        @testset "Histogram Flag Tests" begin
            @test @isdefined do_you_see_the_histogram
            @test isa(do_you_see_the_histogram, Bool)
            @test do_you_see_the_histogram == true
        end
        
        # Test 4: HelloWorld Function Tests
        @testset "HelloWorld Function Tests" begin
            @test @isdefined message_that_we_get
            @test isa(message_that_we_get, String)
            @test message_that_we_get == "Hello World!"
            
            # Test the function directly
            greeting = printgreeting()
            @test greeting == "Hello World!"
            @test isa(greeting, String)
        end
        
        # Test 5: Data Type and Structure Tests
        @testset "Data Type and Structure Tests" begin
            # Test that all variables have expected types
            @test isa(samples, Vector)
            @test eltype(samples) == Float64
            
            # Test that samples are finite (no NaN or Inf values)
            @test all(isfinite.(samples))
            
            # Test range - normal distribution should have most values within ±4 standard deviations
            extreme_values = count(abs.(samples) .> 4.0)
            @test extreme_values < 100
        end
        
        # Test 6: Package and Function Availability Tests
        @testset "Package and Function Availability Tests" begin
            @test @isdefined histogram
            @test @isdefined randn
        end

        # Test 7: The tested engineering calculation
        @testset "Ideal Gas Pressure Tests" begin
            @test isapprox(pressure_Pa, 101_325.0; rtol = 1e-8)
            @test ideal_gas_pressure(2, 300, 0.05) isa Float64
            @test_throws ArgumentError ideal_gas_pressure(0, 300, 0.05)
            @test_throws ArgumentError ideal_gas_pressure(1, -10, 0.05)
            @test_throws ArgumentError ideal_gas_pressure(1, 300, Inf)
        end
    end
end;

___

## Summary
A smoke test answers one question before any real work starts: does the toolchain on this machine actually run the course material end to end?

> __Key Takeaways:__
>
> * **One include connects everything:** A single meeting-local `Include.jl` activates the pinned root environment and loads the local source, so a notebook needs no per-machine configuration of its own.
> * **A smoke test exercises every layer:** Running packages, local source, a computation, and a test set together proves that all four work, which no single one of them can establish alone.
> * **A clean run is a baseline:** Once the notebook has run top to bottom, any later failure can be attributed to the change you just made rather than to the installation.

Keep this notebook working. When something breaks later in the semester, re-running it is the fastest way to tell a broken environment from a broken idea.
___